# **Discharge diagnostics**
### Ève Castonguay, LIRA

In [41]:
# This code establishes statistical diagnostics on the discharge DataSet created by the file [discharge_dset.ipynb], and also shows various plots.
# The main goal is to compare the swot discharge data with in-situ values available in the grdc dataset.
# Author: Ève Castonguay, LIRA (CNRS)
# Creation date: 2026-06-11 [YYYY-MM-DD]
# Version 0.1: AAAA-MM-JJ

In [57]:
# Imports
from datetime import datetime
import os
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import glob 
import numpy as np
import netCDF4 as nc
from scipy.spatial import KDTree
from mpl_toolkits.basemap import Basemap
import cartopy.crs as ccrs
import math
from scipy import stats
import matplotlib.colors as mcolors
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import geopy.distance
import random
from datetime import date, timedelta

#### Download discharge dset

In [58]:
dset_version = 'v7_3'
dir_dset = '/obs/ecastonguay/scripts/global_dset_' + dset_version + '.nc'
dset_global = xr.open_dataset(dir_dset, engine="netcdf4") # discharge dataset
print(dset_global)
print('DATA VARIABLES:', list(dset_global.data_vars))

# when finished
# dset_global.close()

<xarray.Dataset> Size: 50MB
Dimensions:       (id: 5286, time: 766)
Coordinates:
  * id            (id) int64 42kB 4101200 4101400 4101451 ... 5870600 5870655
  * time          (time) datetime64[ns] 6kB 2023-03-29 2023-03-30 ... 2025-05-02
Data variables: (12/14)
    dschg_g       (id, time) float32 16MB ...
    geox_g        (id) float64 42kB ...
    geoy_g        (id) float64 42kB ...
    area_g        (id) float64 42kB ...
    river_name_g  (id) <U33 698kB ...
    country_g     (id) <U2 42kB ...
    ...            ...
    geoy_s        (id) float64 42kB ...
    id_s          (id) float64 42kB ...
    width_s       (id) float64 42kB ...
    area_s        (id) float64 42kB ...
    river_name_s  (id) <U32 677kB ...
    wse_s         (id) float64 42kB ...
DATA VARIABLES: ['dschg_g', 'geox_g', 'geoy_g', 'area_g', 'river_name_g', 'country_g', 'dschg_s', 'geox_s', 'geoy_s', 'id_s', 'width_s', 'area_s', 'river_name_s', 'wse_s']


### **Add diagnostics to the dset**

#### 1. Selecting stations with sufficient swot-grdc temporal overlap

In [59]:
## Swot is ~21 days & grdc is ~daily. Both don't have data at the same time.
## Creating DataArrays containing dschg only when both have it.
co = 6 # number of correspondences

# Count the number of days for which there are swot & grdc mesures, per station
mask_coincide = (dset_global.dschg_g.notnull() & dset_global.dschg_s.notnull()) # Array of True/False. True where there are both swot & grdc data. .notnull() is pandas but works with np.nan
n_coincide = mask_coincide.sum(dim="time") # sum of Trues per station (5241,). 0 where there is no coincid. [tested]

# Compute the mean
mean_coincide = n_coincide.where(n_coincide != 0).mean() # 40.77 across all continent, ignoring zeros (WHEN there is a corresp., the avg number of time is ...) [tested]
mean_coincide_null = np.nanmean(n_coincide) # mean of 29.88 across all continents, including zeros [tested]

# Compute the max
max_conincide = np.nanmax(n_coincide) # 183 times

# Select stations where there are Co or more corresp.
ids_co = n_coincide.id.where(n_coincide >= co, drop=True) # get the list of station id where stats will be computed [tested] https://docs.xarray.dev/en/stable/generated/xarray.where.html
assert len(ids_co) != len(n_coincide.id) # since we dropped some ids, len shold be different except if co = 0
q_co_g = dset_global.sel(id=ids_co)["dschg_g"]
q_co_s = dset_global.sel(id=ids_co)["dschg_s"]
mask_coincide_co = mask_coincide.sel(id=ids_co) 

# DataArray of swot & grdc data only where both have data at the same time (rest is nan)
q_ol_g = q_co_g.where(mask_coincide_co) # ol : overlap
q_ol_s = q_co_s.where(mask_coincide_co)

# Retrieve (lat,lon) of stations
x_ol_s = dset_global.sel(id=ids_co)['geox_s'].values
y_ol_s = dset_global.sel(id=ids_co)['geoy_s'].values
x_ol_g = dset_global.sel(id=ids_co)['geox_g'].values
y_ol_g = dset_global.sel(id=ids_co)['geoy_g'].values

# Retrieve width of stations
w_ol_swr = dset_global.sel(id=ids_co)['width_s'].values

# Retrieve areas
area_ol_s = dset_global.sel(id=ids_co)['area_s']
area_ol_g = dset_global.sel(id=ids_co)['area_g']

# Retrieve ids
id_ol_s = dset_global.sel(id=ids_co)['id_s']
id_ol_g = dset_global.sel(id=ids_co).id.values

# List of station ids in the original dset
list_station_id = dset_global.id.values

print(f"Number of stations with overlap higher or equal to {co} : {len(ids_co)}")


Number of stations with overlap higher or equal to 6 : 729


#### 2. Different statistics & add them to the global dset

In [60]:
################################### N coincide
# vector of int to incidate number of temporal matches between a grdc and a reach time series 
n_co_darray = xr.DataArray(
        data=n_coincide,
        dims=["id"],
        coords=dict(
            id=n_coincide.id,
        ),
        attrs=dict(
            description="Incidates the number of temporal matches between a grdc and a reach time series. When the station was not matched, value is 0"
        ),
        name="n_coincide" 
    )
dset_global['n_coincide'] = n_co_darray

################################### Matches
# vector of bool. to locate matches ('True') in the station id list of the dset
matches_darray = xr.DataArray(
        data=np.full(len(ids_co), True, dtype=int),
        dims=["id"],
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="True where there is a match between a reach & station, and where the number of temporal correspondence is equal or superior to 6 (for v7) (see [diagnostics_dset.ipynb]). Else is False"

        ),
        name="match" 
    )
matches_darray = matches_darray.reindex(id=list_station_id, fill_value=False)
dset_global['match'] = matches_darray

################################### Bias on discharge (daily mean error), %
eps = 0.5 # to filter out the smaller discharges that would result in a value of inf (/0)
inf_mask = (q_ol_s >= eps) & (q_ol_g >= eps)
q_mask_g = q_ol_g.where(inf_mask) 
q_mask_s = q_ol_s.where(inf_mask) 
daily_err = (q_mask_s - q_mask_g) / q_mask_g # Nans will be ignored in the computations
bias = daily_err.mean(dim="time")*100
# Create DataArray
q_bias_darray = xr.DataArray(
        data=bias,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Swot bias (mean relative error on the discharge time series, %)" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates # *** this is done automatically in the next line
q_bias_darray = q_bias_darray.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['q_bias_d'] = q_bias_darray

################################### MAE 
daily_err = np.absolute(q_ol_s-q_ol_g)
mae = daily_err.mean(dim="time")
# Create DataArray
q_mae_darray = xr.DataArray(
        data=mae,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Swot mean absolute error (MAE) on discharge" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
q_mae_darray = q_mae_darray.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['q_mae_d'] = q_mae_darray

################################### Temporal correlation (Pearson)
# Measuring temp. corr. only when grdc & swot both have data at a certain time 
r_matrix = xr.corr(q_ol_s, q_ol_g, dim='time') # [tested]
# Create DataArray
pearson_darray = xr.DataArray(
        data=r_matrix,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Pearson correlation coefficient on discharge" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
pearson_darray = pearson_darray.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['pearson_corr_d'] = pearson_darray

################################### KGE
def kge(sim, obs):
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    mask = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[mask]
    obs = obs[mask]

    r = np.corrcoef(sim, obs)[0, 1]     # this is a correlation matrix. has size (2,2) for the 2 variables that are put as rows and columns. 
                                        # diagonals should always equal 1 because a given variable is perfectly correlated with itself.
    alpha = np.std(sim) / np.std(obs)
    beta = np.mean(sim) / np.mean(obs)

    kge_value = 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

    return kge_value #, r, alpha, beta
assert q_ol_g.shape == q_ol_s.shape
# Compute statistic
kge_matrix = np.array([ # q_ol_g.shape: (2776, 766)
    kge(q_ol_s[ii, :], q_ol_g[ii, :])
    for ii in range(q_ol_g.shape[0])
])
if kge_matrix.shape[0] != q_ol_g.shape[0] != q_ol_s.shape[0]:
    print("Error: Matrixes should have same number of rows. If the kge function deleted rows due to infinite values, adjust (lat,lon) accordingly.")
# Create DataArray
kge_darray = xr.DataArray(
        data=kge_matrix,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Kling–Gupta Efficiency (KGE) coefficient on discharge" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
kge_darray = kge_darray.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['kge_d'] = kge_darray
    
################################### SWOT amplitude
min_s = q_ol_s.min(dim="time",skipna=True)
max_s = q_ol_s.max(dim="time",skipna=True)
ampli_s = max_s-min_s
# Create DataArray
ampli_darray_s = xr.DataArray(
        data=ampli_s,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Swot amplitude (max-min of the discharge time series)" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
ampli_darray_s = ampli_darray_s.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['ampli_s'] = ampli_darray_s

################################### GRDC amplitude
min_g = q_ol_g.min(dim="time",skipna=True)
max_g = q_ol_g.max(dim="time",skipna=True)
ampli_g = max_g-min_g
# Create DataArray
ampli_darray_g = xr.DataArray(
        data=ampli_g,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Grdc amplitude (max-min of the discharge time series)" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
ampli_darray_g = ampli_darray_g.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['ampli_g'] = ampli_darray_g

################################### Amplitude percentage
ampli_pct = (ampli_s / ampli_g) * 100
ampli_darray_gs = xr.DataArray(
        data=ampli_pct,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Percentage of the grdc amplitude accounted for by swot amplitude; (A_s/A_g)*100" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
ampli_darray_g = ampli_darray_g.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['ampli_pct_d'] = ampli_darray_gs

################################### Distance between GRDC and SWT coordinates
distance = np.full(len(y_ol_s),np.nan, dtype=np.float64)
for jj in range (len(y_ol_s)):
    coords_s = (y_ol_s[jj], x_ol_s[jj])
    coords_g = (y_ol_g[jj], x_ol_g[jj])
    distance[jj] = geopy.distance.geodesic(coords_s, coords_g).km
# Create DataArray
dist_darray_g = xr.DataArray(
        data=distance,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Distance (km) between the grdc station and the selected swot reach" 
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
dist_darray_g = dist_darray_g.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['distance_d'] = dist_darray_g

################################### NSE 
def compute_nse(sim, obs): # *** what happens if nans???
    sim = np.asarray(sim, dtype=float)
    obs = np.asarray(obs, dtype=float)

    mask = np.isfinite(sim) & np.isfinite(obs)
    sim = sim[mask]
    obs = obs[mask]

    mse = np.mean((sim - obs)**2) # or : mean_squared_error(obs, sim)
    var_obs = np.std(obs)**2

    nse_value = 1 - mse/var_obs

    return nse_value #, r, alpha, beta
assert q_ol_g.shape == q_ol_s.shape
# Compute statistic
nse_array = np.array([ # q_ol_g.shape: (2776, 766)
    compute_nse(q_ol_s[nn, :], q_ol_g[nn, :])
    for nn in range(q_ol_g.shape[0])
])
if nse_array.shape[0] != q_ol_g.shape[0] != q_ol_s.shape[0]:
    print("Error: Matrixes should have same number of rows. If the kge function deleted rows due to infinite values, adjust (lat,lon) accordingly.")
# Create DataArray
nse_darray_g = xr.DataArray(
        data=nse_array,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Nash–Sutcliffe efficiency (NSE) coefficient on discharge"
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
nse_darray_g = nse_darray_g.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['nse_d'] = nse_darray_g

################################### Bias on areas
inf_mask = (area_ol_s >= eps) & (area_ol_g >= eps)
a_mask_g = area_ol_g.where(inf_mask) 
a_mask_s = area_ol_s.where(inf_mask) 
area_bias = ((a_mask_s - a_mask_g) / a_mask_g)*100
# Create DataArray
area_bias_darray_g = xr.DataArray(
        data=area_bias,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="[Insert description]"
        ),
        name="Bias (relative error, %) on the areas"
    )
"""# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
area_bias_darray_g = area_bias_darray_g.reindex(id=list_station_id)"""
# Put in global DataSet
dset_global['area_bias_d'] = area_bias_darray_g

################################### Mean swot discharge for a station
# Create DataArray
mean_q_darray_s = xr.DataArray(
        data=dset_global['dschg_s'].mean(dim='time', skipna=True),
        dims=["id"], 
        coords=dict(
            id=dset_global.id,
        ),
        attrs=dict(
            description="Mean swot discharge for a given station (m^3/s), over the entire time series"
        ),
        name="mean_dschg_s"
    )
# Put in global DataSet
dset_global['mean_dschg_s'] = mean_q_darray_s

################################### Mean grdc discharge for a station
# Create DataArray
mean_q_darray_g = xr.DataArray(
        data=dset_global['dschg_g'].mean(dim='time', skipna=True),
        dims=["id"], 
        coords=dict(
            id=dset_global.id,
        ),
        attrs=dict(
            description="Mean grdc discharge for a given reach (m^3/s), over the entire time series"
        ),
        name="mean_dschg_g"
    )
# Put in global DataSet
dset_global['mean_dschg_g'] = mean_q_darray_g

################################### Seasonnal anomalies (Filipe had a better name for it, don't remember)
# initial idea: comparer max-min (sur toutes les 2 anneées) de swot et grdc -> valeur relative (?) 

################################### Seasonnal anomalies 2 (Filipe had a better name for it, don't remember)
# Idea: moving average on 3 months for swot & grdc, then compare each temporal distribution with its average
n_days = 90 
swot_start = '2023-03-29'
swot_end = '2025-05-02'
time_dim = pd.date_range(start=swot_start, end=swot_end, freq='D')
assert len(q_ol_g) == len(q_ol_s)
# Compute rolling average
season_swot = n_days//21 # 21 is swot's repeat cycle. // for int of the division
list_station_ids = q_ol_g.id.values
dict_unique_g = {}
dict_unique_s = {}
for sid in list_station_ids:
    q_station_g = q_ol_g.sel(id=sid).dropna("time")
    q_station_s = q_ol_s.sel(id=sid).dropna("time")
    # roll
    rolling_avg_g_unique = q_station_g.rolling(time=season_swot, min_periods=1, center=True).mean().dropna("time") # min_periods=1,
    rolling_avg_s_unique = q_station_s.rolling(time=season_swot, min_periods=1, center=True).mean().dropna("time") # same season beacause q_ol_s is grdc dschg only when swot has data
    # compute anomalies
    assert len(q_station_g) == len(rolling_avg_g_unique) # NOT TRUE if min_periods!=1
    anomalies_g = q_station_g - rolling_avg_g_unique
    anomalies_g = anomalies_g.reindex(time=time_dim)
    anomalies_g = anomalies_g.reindex(id=list_station_id) # "Rolling average (over 3 months) of grdc discharge"
    assert len(q_station_s) == len(rolling_avg_s_unique) # NOT TRUE if min_periods!=1
    anomalies_s = q_station_s - rolling_avg_s_unique
    anomalies_s = anomalies_s.reindex(time=time_dim)
    anomalies_s = anomalies_s.reindex(id=list_station_id) # "Rolling average (over 3 months) of swot discharge"
    # put unique in dict
    dict_unique_g[sid] = anomalies_g
    dict_unique_s[sid] = anomalies_s
# concathenate
rolling_avg_g = xr.concat(list(dict_unique_g.values()), dim='id')
rolling_avg_s = xr.concat(list(dict_unique_s.values()), dim='id')
# Put in global DataSet
dset_global['seasonal_ano_g'] = rolling_avg_g
dset_global['seasonal_ano_s'] = rolling_avg_s

################################### NSE of swot compared to seasonality of grdc
# Find seasonality of GRDC (mean q for a given day)
seasonality_grdc = dset_global['dschg_g'].groupby(['time.month', 'time.day']).mean() # Source - https://stackoverflow.com/a/79931499
# Create continuous time series from the seasonnality
target_months = xr.DataArray(time_dim.month, dims=['time'], coords={'time': time_dim}) # array of 1...12
target_days = xr.DataArray(time_dim.day, dims=['time'], coords={'time': time_dim}) # array of 1...31
seasonality_continuous = seasonality_grdc.sel(
    month=target_months,
    day=target_days
) # with advanced indexing, xarray makes the new darray seasonality_continuous have a time dimension instead of the time.month & time.day. this is because both the time & month was specified and seasonality_grdc initially had this time dimension (time_dim.month, time_dim.day)
# Select seasonality only on stations where there is a match and mask the parts of the time-series where the temporal overlap is <co
seasonality_continuous = seasonality_continuous.sel(id=ids_co).where(mask_coincide_co)
# Compute NSE
nse_array2 = np.array([
    compute_nse(q_ol_s[nn, :], seasonality_continuous[nn, :])
    for nn in range(q_ol_s.shape[0])
])
if nse_array2.shape[0] != seasonality_continuous.shape[0] != q_ol_s.shape[0]:
    print("Error: Matrices should have same number of rows. If the nse function deleted rows due to infinite values, adjust (lat,lon) accordingly (?).")
# Create DataArray
seasonality_darray_g = xr.DataArray(
        data=nse_array2,
        dims=["id"], # name of the dimensions
        coords=dict(
            id=ids_co,
        ),
        attrs=dict(
            description="NSE of swot discharge versus grdc's seasonality (mean value of q on a specific day of the year)"
        ),
        name="nse_seasonality_d"
    )
# Reindex, as we want the darray to have the same station_id list as the global_dset's coordinates
seasonality_darray_g = seasonality_darray_g.reindex(id=list_station_id)
# Put in global DataSet
dset_global['nse_g_seasonality_d'] = seasonality_darray_g


/obs/ecastonguay/miniconda3/envs/swot-env/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/obs/ecastonguay/miniconda3/envs/swot-env/lib/python3.14/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/tmp/ipykernel_9404/1641606050.py:108: RuntimeWarning: divide by zero encountered in scalar divide
  alpha = np.std(sim) / np.std(obs)
/tmp/ipykernel_9404/1641606050.py:109: RuntimeWarning: divide by zero encountered in scalar divide
  beta = np.mean(sim) / np.mean(obs)
/tmp/ipykernel_9404/1641606050.py:234: RuntimeWarning: divide by zero encountered in scalar divide
  nse_value = 1 - mse/var_obs


#### 3. Rename variables (if necessary)

In [19]:
var_name_dict = {"dschg_global_g":"dschg_g",
                 "geox_global_g":"geox_g",
                 "geoy_global_g":"geoy_g",
                 "area_global_g":"area_g",
                 "river_global_g":"river_name_g",
                 "country_global_g":"country_g",
                 "dschg_global_s":"dschg_s",
                 "geox_global_s":"geox_s",
                 "geoy_global_s":"geoy_s",
                 "id_global_s":"id_s",
                 "width_global_s":"width_s",
                 "area_global_s":"area_s",
                 "river_global_s":"river_name_s"} # old : new
dset_global = dset_global.rename_vars(var_name_dict)
print(dset_global)
print('DATA VARIABLES:', list(dset_global.data_vars))

<xarray.Dataset> Size: 100MB
Dimensions:         (time: 766, id: 5286)
Coordinates:
  * time            (time) datetime64[ns] 6kB 2023-03-29 ... 2025-05-02
  * id              (id) int64 42kB 4101200 4101400 4101451 ... 5870600 5870655
Data variables: (12/29)
    dschg_g         (id, time) float32 16MB 1.416 1.416 1.416 ... 59.33 49.56
    geox_g          (id) float64 42kB ...
    geoy_g          (id) float64 42kB ...
    area_g          (id) float64 42kB ...
    river_name_g    (id) <U33 698kB ...
    country_g       (id) <U2 42kB ...
    ...              ...
    nse_d           (id) float64 42kB 0.06868 nan 0.1082 nan ... nan nan nan nan
    area_bias_d     (id) float64 42kB 2.175 nan 6.615 nan ... nan nan nan nan
    mean_dschg_s    (id) float64 42kB 26.14 nan 90.47 16.29 ... nan nan nan nan
    mean_dschg_g    (id) float32 21kB 40.96 60.9 57.05 nan ... 14.67 357.7 59.83
    seasonal_ano_g  (id, time) float32 16MB nan nan nan nan ... nan nan nan nan
    seasonal_ano_s  (id, time) fl

#### 4. Save the new dataset

In [61]:
new_version = 'v7_3'
dset_global.to_netcdf("/obs/ecastonguay/scripts/diag_dset_" + new_version + ".nc")

### **Tests**

#### River names

In [14]:
df = dset_global[['river_global_g', 'id_global_s', 'river_global_s']].isel(id=slice(500, 1000)).to_dataframe()
df['id_global_s'] = df['id_global_s'].astype('Int64')
print(df.to_string()) 

                            river_global_g  id_global_s                    river_global_s
id                                                                                       
4126851                        PEASE RIVER  74227800031                       pease river
4127100                      MERAMEC RIVER  74270800041                     meramec river
4127150                      HATCHIE RIVER         <NA>                                  
4127200                    KASKASKIA RIVER  74270600061                   kaskaskia river
4127201                    KASKASKIA RIVER         <NA>                                  
4127202                    KASKASKIA RIVER         <NA>                                  
4127400                        OBION RIVER  74258000081                       obion river
4127501                  MISSISSIPPI RIVER  74270100061                 mississippi river
4127502                  MISSISSIPPI RIVER  74270500051                 mississippi river
4127503   

#### Number of stations matched with a reach (doesn't garantee that there is temporal overlap with it's station)

In [7]:
r_ids = dset_global['id_global_s'].values 
sum_match = np.nansum(~np.isnan(r_ids))
sum_stations = len(r_ids)
print(f"{sum_match} matches for {sum_stations} stations")

# v3 : 807 matches for 5286 stations (thr = [0.5, 20, 75])
# v4 : 356 matches for 5286 stations
# v5 : 394 matches for 5286 stations (thr = [0.5, 20, 75, 50])
# v5_1 : 555 matches for 5286 stations (thr = [0.5, 20, 75, 100])
# v6 : 922 matches for 5286 stations
# v6_1 : 922 matches for 5286 stations
# v7 : 783 matches for 5286 stations (thr = [0.5, 20, 90, 50]) *
# v7_1 : 814 matches for 5286 stations (thr = [0.5, 20, 90, 100]) 
# v7_2 : 840 matches for 5286 stations (thr = [0.5, 20, 80, 50]) *
# v8 matt : 483 matches for 5286 stations (thr = [0.5, 10, 90, 10]) 
# v8_1 : 922 matches for 5286 stations (thr = [0.5, 20, 75, 100]) 

783 matches for 5286 stations


#### Station ids of the biggest distances (for later on: take the reaches as far as 100km but sort visually)

In [ ]:
# step 1: change the vX to the version with matches as far as 100km
#idx_top10 = np.argpartition(distance, -10)[-10:]
#print("Top 10 biggest distances",ids_co[idx_top10].values)  # [4120902., 6337515., 6545050., 4122600., 4115201., 4119600., 3650150., 4207160., 1159301., 1159304.]
                                                            # after speaking with Victor: eliminate 4119600, 4207160
                                                            # then plot time series and check on google maps

"""max_dist_r = np.argmax(distance)
print(ids_co[max_dist_r]) # 6605440
print(np.max(distance)) # 48 km
single_s = dset_global[['river_global_g', 'geox_global_g', 'geoy_global_g', 'area_global_g', 'id_global_s', 'geox_global_s', 'geoy_global_s', 'area_global_s', 'river_global_s']].sel(id=[1159304]).to_dataframe()
single_s['id_global_s'] = single_s['id_global_s'].astype('Int64')
print(single_s.to_string()) """

#### View diagnostics

In [ ]:
df = dset_global[['q_bias_d', 'id_global_s', 'river_global_s']].isel(id=slice(500, 1000)).to_dataframe()
df['id_global_s'] = df['id_global_s'].astype('Int64')
print(df.to_string()) 

#### Test on one value

In [ ]:
id = 4126851
q_g = dset_global.sel(id=id)["dschg_global_g"]
q_s = dset_global.sel(id=id)["dschg_global_s"]
eps = 0.5 # to filter out the smaller discharges that would result in a value of inf (/0)
inf_mask = (q_s >= eps) & (q_g >= eps)
q_mask_g = q_g.where(inf_mask) 
q_mask_s = q_s.where(inf_mask) 
daily_err = (q_mask_s - q_mask_g) / q_mask_g # Nans will be ignored in the computations
bias = daily_err.mean(dim="time")*100
print(bias)

#### Stations in caravan & the matched pairs in eve

In [15]:
dir = '/obs/ecastonguay/scripts/in_caravan_in_eve.csv'
df = pd.read_csv(dir)
df["gauge_id_num"] = df["gauge_id"].str.extract(r"GRDC_(\d+)").astype(int)

# number of values that intersect - 661
nb1 = len(set(ids_co.values) & set(df["gauge_id_num"]))
print(nb1)

# values that intersect
intst = set(ids_co.values).intersection(set(df["gauge_id_num"]))

661
